# Embedding + Tabular: E1〜E6の同一時系列CV比較

保存済みEmbeddingと表形式特徴を、既存のexpanding-window foldsで比較します。前処理とPCAは各foldのtrainingだけでfitします。初期状態では学習しません。

T4 x1を利用する既定設定です。E3/E4はPyTorch CUDA + mixed precision、E5はCatBoost GPU、E6はXGBoost CUDAを使います。

In [ ]:
from pathlib import Path
import logging
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from embedding_features import load_embeddings
from modeling import (
    default_modeling_config,
    fit_full_and_predict_test,
    run_all_experiments,
)
from validation import make_time_series_cv

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

## 1. データと特徴列

列名は実データに合わせて編集してください。`project_start_year == -1`は既存CV関数がfoldから除外し、表特徴側でもmissingとして扱います。

In [ ]:
TARGET_COL = 'target'
YEAR_COL = 'project_start_year'
PROJECT_COL = 'project_name'
PROJECT_ID_COL = 'project_id'

NUMERIC_COLS = [
    'project_start_year',
    'project_end_year',
    'project_fiscal_year',
    'budget',
]
CATEGORICAL_COLS = [
    'responsible_ministry',
]

train = pd.read_csv(PROJECT_ROOT / 'input' / 'train.csv')
test = pd.read_csv(PROJECT_ROOT / 'input' / 'test.csv')
print('train:', train.shape, 'test:', test.shape)

## 2. Embedding cacheを明示的に選択

OpenAI/Geminiの切り替えは`EMBEDDING_CACHE_DIR`だけで行います。候補を表示してから、train/testの両方が入った同じdirectoryを指定してください。行順はmetadataの`project_id`と元indexで検証されます。

In [ ]:
EMBEDDING_ROOT = PROJECT_ROOT / 'data' / 'embeddings'
available_caches = sorted(path for path in EMBEDDING_ROOT.glob('*') if path.is_dir())
for path in available_caches:
    print(path)

# 例: EMBEDDING_CACHE_DIR = available_caches[0]
EMBEDDING_CACHE_DIR = None

In [ ]:
train_embeddings = test_embeddings = None
train_embedding_metadata = test_embedding_metadata = None
if EMBEDDING_CACHE_DIR is not None:
    train_embeddings, train_embedding_metadata = load_embeddings(
        EMBEDDING_CACHE_DIR,
        split='train',
        expected_df=train,
        project_id_col=PROJECT_ID_COL,
    )
    test_embeddings, test_embedding_metadata = load_embeddings(
        EMBEDDING_CACHE_DIR,
        split='test',
        expected_df=test,
        project_id_col=PROJECT_ID_COL,
    )
    print('train embedding:', train_embeddings.shape)
    print('test embedding :', test_embeddings.shape)
else:
    print('EMBEDDING_CACHE_DIRを選択してください。')

## 3. 全実験で共有する時系列fold

In [ ]:
folds, cv_diagnostics = make_time_series_cv(
    df=train,
    year_col=YEAR_COL,
    project_col=PROJECT_COL,
    target_col=TARGET_COL,
    n_valid_years=3,
)
display(cv_diagnostics)

## 4. T4向けconfig

通常のE6はPCAなしです。`[None, 512, 256]`へ変えた場合だけ追加PCA実験を行います。CatBoost GPUは演算順の都合でbitwise deterministicではありません。

In [ ]:
CONFIG = default_modeling_config()
CONFIG['output_dir'] = str(PROJECT_ROOT / 'outputs')
CONFIG['metric'] = 'roc_auc'  # または 'log_loss'
CONFIG['e6_pca_dims'] = [None]

# T4 x1
CONFIG['mlp']['device'] = 'cuda'
CONFIG['mlp']['batch_size'] = 256
CONFIG['mlp']['use_amp'] = True
CONFIG['catboost']['task_type'] = 'GPU'
CONFIG['catboost']['devices'] = '0'
CONFIG['xgboost']['device'] = 'cuda'
CONFIG['xgboost']['tree_method'] = 'hist'

# 最初はCVだけ。test予測はCV結果を見た後に下の専用セルで実行する。
CONFIG['run_final_test_prediction'] = False
CONFIG

In [ ]:
try:
    import torch
    print('torch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
except ImportError:
    print('torch is not installed. Run: python -m pip install -r requirements.txt')

## 5. E1〜E6を実行

`RUN_CV=True`にした場合だけ学習します。すべて同じ`folds`を受け取り、OOFは`outputs/oof_predictions.parquet`へ統一保存されます。

In [ ]:
RUN_CV = False
suite = None
if RUN_CV:
    if train_embeddings is None:
        raise RuntimeError('先にEMBEDDING_CACHE_DIRを選択してください。')
    suite = run_all_experiments(
        train=train,
        folds=folds,
        train_embeddings=train_embeddings,
        train_embedding_metadata=train_embedding_metadata,
        numeric_cols=NUMERIC_COLS,
        categorical_cols=CATEGORICAL_COLS,
        config=CONFIG,
        target_col=TARGET_COL,
        project_col=PROJECT_COL,
        project_id_col=PROJECT_ID_COL,
        year_col=YEAR_COL,
    )
    display(suite.summary.sort_values('mean', ascending=CONFIG['metric'] == 'log_loss'))
else:
    print('CV is disabled. Set RUN_CV=True when ready.')

## 6. fold・seen/unseen・時間・PCAの詳細

In [ ]:
if suite is not None:
    detail_cols = [
        'experiment', 'validation_year', 'overall_score',
        'seen_score', 'unseen_score', 'seen_ratio',
        'fit_seconds', 'predict_seconds', 'input_dim',
        'device', 'best_iteration', 'pca_explained_variance',
    ]
    display(suite.fold_metrics[detail_cols])
    display(suite.oof_predictions.notna().sum().rename('OOF rows').to_frame())

## Optional: E6のPCA比較

必要なときだけ`CONFIG['e6_pca_dims'] = [None, 512, 256]`へ変更してCVセルを再実行します。各foldのPCAはfold training embeddingだけでfitされ、explained varianceがfold metricsへ保存されます。

## 7. 選抜モデルだけtrain全体で再fitしてtest予測

CVを再実行せず、選んだモデルだけを全trainで学習します。`RUN_FINAL_TEST_PREDICTION=True`へ変更するまで実行されません。PCA版を選ぶ場合はexperiment名と次元を対応させてください。

In [ ]:
RUN_FINAL_TEST_PREDICTION = False
FINAL_EXPERIMENTS = {
    # 'E1_embedding_lr': None,
    # 'E6_embedding_tabular_xgb': None,
    # 'E6_embedding_tabular_xgb_pca512': 512,
}

if RUN_FINAL_TEST_PREDICTION:
    if train_embeddings is None or test_embeddings is None:
        raise RuntimeError('先にtrain/test Embeddingを読み込んでください。')
    if not FINAL_EXPERIMENTS:
        raise RuntimeError('CV結果を見てFINAL_EXPERIMENTSを選択してください。')
    prediction_dir = PROJECT_ROOT / 'outputs' / 'test_predictions'
    prediction_dir.mkdir(parents=True, exist_ok=True)
    for experiment, pca_dim in FINAL_EXPERIMENTS.items():
        prediction = fit_full_and_predict_test(
            experiment=experiment,
            train=train,
            test=test,
            train_embeddings=train_embeddings,
            test_embeddings=test_embeddings,
            train_embedding_metadata=train_embedding_metadata,
            test_embedding_metadata=test_embedding_metadata,
            numeric_cols=NUMERIC_COLS,
            categorical_cols=CATEGORICAL_COLS,
            config=CONFIG,
            pca_dim=pca_dim,
            target_col=TARGET_COL,
            project_id_col=PROJECT_ID_COL,
        )
        np.save(prediction_dir / f'{experiment}.npy', prediction.astype(np.float32))
        print(experiment, prediction.shape, prediction.min(), prediction.max())
else:
    print('Final test prediction is disabled.')